In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid")

# 2. Thu thập & Đọc dữ liệu
df = pd.read_csv('xsmn_giai_dac_biet_1thang.csv')
print("--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
print(df.head())
print("\n--- THÔNG TIN DỮ LIỆU ---")
print(df.info())

--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---
         ngay        dai  giai_dac_biet
0  2026-09-25    Cần Thơ          54886
1  2026-09-25  Vĩnh Long         137337
2  2026-09-25     TP.HCM         999890
3  2026-09-26     TP.HCM          64820
4  2026-09-26    Cần Thơ         787201

--- THÔNG TIN DỮ LIỆU ---
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ngay           90 non-null     str  
 1   dai            90 non-null     str  
 2   giai_dac_biet  90 non-null     int64
dtypes: int64(1), str(2)
memory usage: 2.2 KB
None


In [13]:
# Kiểm tra dữ liệu khuyết
print("Số lượng dữ liệu khuyết:")
print(df.isnull().sum())

# Làm sạch cột ngày
df['ngay'] = pd.to_datetime(df['ngay'], errors='coerce')
df = df.dropna(subset=['ngay']).reset_index(drop=True)

# Làm sạch tên đài
df['dai'] = df['dai'].astype(str).str.strip()
df['dai'] = df['dai'].str.replace(r'\s+', ' ', regex=True)

# Làm sạch giải đặc biệt
df['giai_dac_biet'] = df['giai_dac_biet'].astype(str).str.strip()
df['giai_dac_biet'] = df['giai_dac_biet'].str.replace(r'\D', '', regex=True)  # chỉ giữ số
df['giai_dac_biet'] = df['giai_dac_biet'].str.zfill(6)

# Loại bỏ các hàng không hợp lệ (không đủ 6 chữ số)
df = df[df['giai_dac_biet'].str.len() == 6].reset_index(drop=True)

# Tạo các cột mới sau khi đã sạch dữ liệu
df['dau_so'] = df['giai_dac_biet'].str[0].astype(int)
df['hai_so_cuoi'] = df['giai_dac_biet'].str[-2:].astype(str).str.zfill(2)
df['tong_chu_so'] = df['giai_dac_biet'].apply(lambda x: sum(int(digit) for digit in x))

# Sắp xếp theo ngày
df = df.sort_values('ngay').reset_index(drop=True)
df['day_num'] = (df['ngay'] - df['ngay'].min()).dt.days

print("\n--- DỮ LIỆU SAU KHI LÀM SẠCH ---")
print(df.head())
print(df.info())

# Ép kiểu dữ liệu chuỗi 6 chữ số
df['giai_dac_biet'] = df['giai_dac_biet'].astype(str).str.zfill(6)
df['ngay'] = pd.to_datetime(df['ngay'])

# Biến đổi tạo cột mới (Feature Engineering)
df['dau_so'] = df['giai_dac_biet'].str[0].astype(int)
df['hai_so_cuoi'] = df['giai_dac_biet'].str[-2:].astype(int)
df['tong_chu_so'] = df['giai_dac_biet'].apply(lambda x: sum(int(digit) for digit in x))

# Sắp xếp theo ngày
df = df.sort_values('ngay').reset_index(drop=True)
df['day_num'] = (df['ngay'] - df['ngay'].min()).dt.days

print("\n--- DỮ LIỆU SAU KHI BIẾN ĐỔI (FEATURE ENGINEERING) ---")
print(df[['ngay', 'dai', 'giai_dac_biet', 'hai_so_cuoi', 'tong_chu_so', 'day_num']].head())


Số lượng dữ liệu khuyết:
ngay             0
dai              0
giai_dac_biet    0
dau_so           0
hai_so_cuoi      0
tong_chu_so      0
day_num          0
dtype: int64

--- DỮ LIỆU SAU KHI LÀM SẠCH ---
        ngay        dai giai_dac_biet  dau_so hai_so_cuoi  tong_chu_so  \
0 2026-09-25    Cần Thơ        054886       0          86           31   
1 2026-09-25  Vĩnh Long        137337       1          37           24   
2 2026-09-25     TP.HCM        999890       9          90           44   
3 2026-09-26     TP.HCM        064820       0          20           20   
4 2026-09-26    Cần Thơ        787201       7          01           25   

   day_num  
0        0  
1        0  
2        0  
3        1  
4        1  
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ngay           90 non-null     datetime64[us]
 1   dai            90 non-n

In [9]:
# Nhóm dữ liệu theo Đài
summary_by_dai = df.groupby('dai').agg(
    so_lan_mo_thuong=('giai_dac_biet', 'count'),
    trung_binh_tong_digit=('tong_chu_so', 'mean'),
    min_2so_cuoi=('hai_so_cuoi', 'min'),
    max_2so_cuoi=('hai_so_cuoi', 'max')
).reset_index().sort_values(by='so_lan_mo_thuong', ascending=False)

print("--- BẢNG THỐNG KÊ THEO ĐÀI ---")
print(summary_by_dai)


--- BẢNG THỐNG KÊ THEO ĐÀI ---
          dai  so_lan_mo_thuong  trung_binh_tong_digit  min_2so_cuoi  \
5    Tây Ninh                16              30.375000            23   
2     Cần Thơ                15              29.600000             1   
3      TP.HCM                13              31.538462             9   
7    Đồng Nai                12              30.916667             6   
0    An Giang                10              27.700000             3   
6   Vĩnh Long                 9              29.222222            14   
1  Bình Dương                 8              28.375000            31   
4  Tiền Giang                 7              25.428571            11   

   max_2so_cuoi  
5            98  
2            91  
3            92  
7            95  
0            86  
6            92  
1            95  
4            94  


## Câu hỏi thống kê

1. Hai chữ số cuối nào xuất hiện nhiều nhất trong mẫu?
2. Những cặp chữ số nào chưa xuất hiện trong mẫu?
3. Mô hình đơn giản có vượt baseline trên phần dữ liệu kiểm tra theo thời gian không?

Tần suất trong một mẫu nhỏ chỉ mô tả mẫu đó, không chỉ ra số nào sẽ xuất hiện tiếp theo.

In [ ]:
# Đếm số lần xuất hiện của từng cặp số 00 - 99 (cặp chưa từng xuất hiện được tính 0 lần)
tan_suat = df['hai_so_cuoi'].value_counts().reindex(range(100), fill_value=0)
bang = tan_suat.rename_axis('cap_so').reset_index(name='so_lan')
bang['cap_so'] = bang['cap_so'].astype(str).str.zfill(2)
bang = bang.sort_values(['so_lan', 'cap_so'], ascending=[False, True]).reset_index(drop=True)

print("Tần suất hai chữ số cuối:")
display(df["hai_so_cuoi"].value_counts().head(15))


chua_ra = sorted(bang[bang['so_lan'] == 0]['cap_so'].tolist())
print(f"\nCặp số chưa xuất hiện: {len(chua_ra)}/100")
print(chua_ra)

print("\nSố dòng theo nhãn địa phương mô phỏng:")
display(df["province"].value_counts())

Tần suất hai chữ số cuối:


hai_so_cuoi
79    5
39    4
48    4
86    3
37    3
11    3
76    3
90    2
38    2
36    2
95    2
89    2
32    2
46    2
70    2
Name: count, dtype: int64


Cặp số chưa xuất hiện: 40/100
['00', '02', '05', '07', '08', '10', '12', '16', '17', '18', '21', '26', '27', '28', '30', '34', '40', '41', '42', '44', '45', '47', '55', '57', '59', '60', '62', '65', '66', '67', '68', '73', '77', '78', '80', '88', '93', '96', '97', '99']


In [ ]:
plt.figure(figsize=(14, 5))

# Biểu đồ 1: Histplot phân bố 2 số cuối
plt.subplot(1, 2, 1)
sns.histplot(df['hai_so_cuoi'], bins=10, kde=True, color='skyblue', edgecolor='black')
plt.title('Biểu đồ 1: Phân bố 2 số cuối Giải Đặc Biệt', fontsize=12, fontweight='bold')
plt.xlabel('2 Số cuối (00 - 99)')
plt.ylabel('Tần suất')

# Biểu đồ 2: Lineplot xu hướng Tổng chữ số
plt.subplot(1, 2, 2)
sns.lineplot(data=df, x='day_num', y='tong_chu_so', marker='o', color='crimson')
plt.title('Biểu đồ 2: Biến động Tổng chữ số GĐB theo thời gian', fontsize=12, fontweight='bold')
plt.xlabel('Ngày thứ (kể từ mốc bắt đầu)')
plt.ylabel('Tổng chữ số GĐB')

plt.tight_layout()
plt.show()


In [ ]:
# Chuẩn bị dữ liệu huấn luyện
X = df[['day_num']]  # Biến độc lập: Ngày
y = df['tong_chu_so'] # Biến phụ thuộc: Tổng chữ số GĐB

# Chia tập Train - Test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Huấn luyện mô hình Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# Dự đoán trên tập test
y_pred = model.predict(X_test)

# Đánh giá mô hình
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- ĐÁNH GIÁ MÔ HÌNH HỒI QUY TUYẾN TÍNH ---")
print(f"Hệ số góc: {model.coef_[0]:.4f}")
print(f"Hệ số chặn: {model.intercept_:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R^2: {r2:.4f}")


---

## 📌 BUỔI 12: CÂU CHUYỆN DỮ LIỆU, TỰ ĐÁNH GIÁ VÀ HƯỚNG PHÁT TRIỂN

### 1. Thống kê & Ý nghĩa Câu chuyện Dữ liệu (Data Story)
* **Kết quả quan sát:** 
  - Các kết quả giải đặc biệt có tính ngẫu nhiên cao, các con số trải đều từ 00 đến 99.
  - Chỉ số $R^2 \approx 0$ (hoặc âm nhẹ trên tập test) phản ánh đúng bản chất ngẫu nhiên độc lập của các kỳ quay xổ số.
* **Kết luận thực tiễn:** Kết quả xổ số là hoàn toàn ngẫu nhiên và độc lập giữa các ngày.

### 2. Tự đánh giá dự án & Hướng phát triển (Self-Evaluation & Future Plan)
* **Điểm mạnh:** 
  - Thực hiện đúng và đủ quy trình Khoa học dữ liệu: Thu thập -> Làm sạch -> Biến đổi -> Phân tích -> Trực quan hóa -> Mô hình hóa.
  - Cấu trúc bài làm sạch sẻ, quản lý module tốt.
* **Kế hoạch phát triển hoàn thiện:**
  - Thu thập dữ liệu dài hạn hơn từ `xosodaiphat.com`.
  - Áp dụng các mô hình Phân loại (Classification) dự đoán tần suất xuất hiện (Đầu/Đuôi/Lô gan) hoặc giao diện Streamlit interactive.
